# 

Hierarchical BCF Fits on Real Data (Dictator + Trust)

Tristan Muno [](https://orcid.org/0009-0002-3078-8436) (University of Mannheim)  
Vera Okisheva (University of Mannheim)  
Raphael Klöckner (University of Mannheim)  
June 12, 2026

# Purpose

This notebook fits the **robustness** estimator — hierarchical BCF via the local `multibart` package — on the full real data for both game outcomes. Scope is **fitting and saving only**: each fit is written to `data/03_final/` for the later post-processing step. No τ(x) extraction, projection, variance-component summaries, figures, or tables are produced here.

Following the resolved feasibility gate, the model uses **country random intercepts only** (the nested respondent-within-country fit was benchmarked and found infeasible; see `03_multibart_nested_ri_test.qmd`, `#sec-purpose`). Within-respondent dependence across the three conjoint rounds is left for a respondent cluster bootstrap in the post-processing step, not modelled here. The variable definitions are reused verbatim from the validated real-data section of `03_multibart_nested_ri_test.qmd` (`#sec-subsample`); the only change is fitting on the full per-game sample rather than a 10,000-row subsample.

> **Note**
>
> The full-sample BCF fits are the expensive step (each game is ~85k rows with `nburn + nsim = 2000`). Quarto `freeze: auto` caches execution at the file level. Do **not** add knitr per-chunk `cache: true` here: the fitted forest’s `tree_samples` is a live Rcpp external pointer that does not survive cache serialize/reload.

# Setup

The setup chunk loads the project packages plus `multibart`’s own dependencies (`Rcpp`, `RcppArmadillo`, `Rcereal`, `fastDummies`) before the local package is touched.

In [ ]:
# execution time
start_time <- Sys.time()
# console width
options(width = 80)
# packages
p_required <- c(
  "tidyverse", # dplyr, ggplot2, tidyr
  "here", # relative paths
  "sessioninfo", # session docs

  # multibart dependencies -----------------------------------------------------
  "Rcpp", # multibart Depends + LinkingTo
  "RcppArmadillo", # multibart Depends + LinkingTo
  "Rcereal", # multibart Depends + LinkingTo
  "fastDummies" # multibart helper dependency
)
packages <- rownames(installed.packages())
p_to_install <- p_required[!(p_required %in% packages)]
if (length(p_to_install) > 0) {
  install.packages(p_to_install)
}
sapply(p_required, require, character.only = TRUE)


Loading required package: tidyverse

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.6
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.1     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: here

here() starts at C:/R/research/MBEU25

Loading required package: sessioninfo

Loading required package: Rcpp

Loading required package: RcppArmadillo

Loading required package: Rcereal

Loading required package: fastDummies

    tidyverse          here   sessioninfo          Rcpp RcppArmadillo 
         TRUE          TRUE          TRUE          TRUE          TRUE 
      Rcereal   fastDummies 
         TRUE          TRUE 

## Local multibart package

`multibart` lives only as source under `code/multibart`; it is installed once (compiling its C++) then attached with `library()`, which is robust for both interactive use and `quarto render`.

In [ ]:
# install local source on first use
if (!requireNamespace("multibart", quietly = TRUE)) {
  install.packages(
    here::here("code", "multibart"),
    repos = NULL,
    type = "source"
  )
}
# attach installed package
library(multibart)


# Load data

In [ ]:
dat <- readRDS(here::here("data", "02_processed", "eu25_long.rds"))


# Variable definitions

The frozen moderator list: profile-level conjoint attributes plus respondent-level pre-treatment items (items and indices together), exactly as in `03_multibart_nested_ri_test.qmd`.

In [ ]:
prof_mods <- c(
  "cj_nationality_shown",
  "der_cj_eu_identity",
  "der_cj_partisanship",
  "cj_age",
  "cj_sex",
  "cj_class"
)
resp_mods <- c(
  "q_gender",
  "q_age",
  "q_identity_country",
  "q_identity_eu",
  "q_identity_europe",
  "q_religion",
  "q_class",
  "q_eu_efficacy_understand",
  "q_pop_reps",
  "q_pop_goodevil",
  "q_pop_compromise",
  "q_dem_compromise",
  "q_dem_listen",
  "q_tech_experts",
  "q_tech_leaders",
  "q_party_harm",
  "q_people_incompetent",
  "q_eu_longterm",
  "q_eu_responsive",
  "q_eu_imp_nat_econ",
  "q_eu_imp_nat_cul",
  "q_eu_imp_nat_pol",
  "q_eu_abolish",
  "q_eu_satisfaction",
  "q_rural_urban",
  "q_edu_age_stop"
)


`prep_game()` assembles the design for one game: outcome, treatment, IDs, and the numeric moderator matrix.

In [ ]:
prep_game <- function(dat, game) {
  sub <- dat |>
    filter(cj_game_type == game) # full per-game sample (no subsampling)

  y <- sub$cj_pl2
  z <- as.integer(sub$cj_rel == "muslim") # Muslim vs all-else pooled
  country_id <- factor(sub$meta_country)
  respondent_id <- factor(sub$meta_pid)

  mod_df <- sub |>
    select(all_of(c(prof_mods, resp_mods))) |>
    mutate(across(where(is.character), factor))
  sub_X <- model.matrix(~ . - 1, data = mod_df) # factors -> dummies

  stopifnot(
    !anyNA(sub_X),
    all(z %in% c(0L, 1L)),
    !anyNA(y),
    nrow(sub_X) == length(y)
  )

  list(
    y = y,
    z = z,
    sub_X = sub_X,
    country_id = country_id,
    respondent_id = respondent_id
  )
}


`fit_bcf()` fits one country-only random-intercept BCF: the `n x C` country-dummy design is the random-effects block, with a single variance component and known conjoint propensity.

In [ ]:
fit_bcf <- function(d) {
  # country-only random-intercept design, built inline
  # (the random_intercepts() helper has a known typo bug; get_dummies() is fine)
  country_dummies <- get_dummies(d$country_id) # n x C
  C <- ncol(country_dummies)
  Q_country <- matrix(1, nrow = C, ncol = 1) # single variance component

  stopifnot(
    nrow(country_dummies) == length(d$y),
    C == nlevels(d$country_id),
    C > 1 # else sampler disables REs
  )

  bcf_continuous_linear(
    y = d$y,
    z = d$z,
    x_control = d$sub_X, # numeric model matrix of moderators
    x_moderate = d$sub_X,
    zhat = rep(0.25, length(d$y)), # known conjoint propensity
    randeff_design = country_dummies,
    randeff_variance_component_design = Q_country,
    randeff_scales = 1, # length K = 1 (one component)
    randeff_df = 3,
    nburn = 1000,
    nsim = 1000,
    update_interval = 100
  )
}


# Dictator-game fit

In [ ]:
d_dict <- prep_game(dat, "cj_dict")
fit_dict <- fit_bcf(d_dict)


Warning in bcf_continuous_linear(y = d$y, z = d$z, x_control = d$sub_X, : All
values of zhat are equal. zhat will not be included among covariates

Warning in regularize.values(x, y, ties, missing(ties), na.rm = na.rm):
collapsing to unique 'x' values
Warning in regularize.values(x, y, ties, missing(ties), na.rm = na.rm):
collapsing to unique 'x' values
Warning in regularize.values(x, y, ties, missing(ties), na.rm = na.rm):
collapsing to unique 'x' values
Warning in regularize.values(x, y, ties, missing(ties), na.rm = na.rm):
collapsing to unique 'x' values

Using random effects.
Reading in y

Setting up designs

design 0
group setting: 0
Instantiated covariate matrix 1 with 55 columns
design 1
group setting: 0
Instantiated covariate matrix 2 with 55 columns
Setting up forests

Done.

Beginning MCMC:
iteration: 0 sigma/SD(Y): 1
iteration: 100 sigma/SD(Y): 0.963919
iteration: 200 sigma/SD(Y): 0.969957
iteration: 300 sigma/SD(Y): 0.96461
iteration: 400 sigma/SD(Y): 0.96468
iteration: 500 sigma/SD(Y): 0.958094
iteration: 600 sigma/SD(Y): 0.962373
iteration: 700 sigma/SD(Y): 0.964664
iteration: 800 sigma/SD(Y): 0.963847
iteration: 900 sigma/SD(Y): 0.9622
iteration: 1000 sigma/SD(Y): 0.962315
iteration: 1100 sigma/SD(Y): 0.961621
iteration: 1200 sigma/SD(Y): 0.968711
iteration: 1300 sigma/SD(Y): 0.965078
iteration: 1400 sigma/SD(Y): 0.963939
iteration: 1500 sigma/SD(Y): 0.966215
iteration: 1600 sigma/SD(Y): 0.962806
iteration: 1700 sigma/SD(Y): 0.9613
iteration: 1800 sigma/SD(Y): 0.959586
iteration: 1900 sigma/SD(Y): 0.962776
time for loop: 560

BCF dictator fit saved (n = 84603).

# Trust-game fit

In [ ]:
d_trust <- prep_game(dat, "cj_trust")
fit_trust <- fit_bcf(d_trust)


Warning in bcf_continuous_linear(y = d$y, z = d$z, x_control = d$sub_X, : All
values of zhat are equal. zhat will not be included among covariates

Warning in regularize.values(x, y, ties, missing(ties), na.rm = na.rm):
collapsing to unique 'x' values
Warning in regularize.values(x, y, ties, missing(ties), na.rm = na.rm):
collapsing to unique 'x' values
Warning in regularize.values(x, y, ties, missing(ties), na.rm = na.rm):
collapsing to unique 'x' values
Warning in regularize.values(x, y, ties, missing(ties), na.rm = na.rm):
collapsing to unique 'x' values

Using random effects.
Reading in y

Setting up designs

design 0
group setting: 0
Instantiated covariate matrix 1 with 55 columns
design 1
group setting: 0
Instantiated covariate matrix 2 with 55 columns
Setting up forests

Done.

Beginning MCMC:
iteration: 0 sigma/SD(Y): 1
iteration: 100 sigma/SD(Y): 0.969359
iteration: 200 sigma/SD(Y): 0.971548
iteration: 300 sigma/SD(Y): 0.966387
iteration: 400 sigma/SD(Y): 0.971051
iteration: 500 sigma/SD(Y): 0.964604
iteration: 600 sigma/SD(Y): 0.959208
iteration: 700 sigma/SD(Y): 0.959948
iteration: 800 sigma/SD(Y): 0.966527
iteration: 900 sigma/SD(Y): 0.962221
iteration: 1000 sigma/SD(Y): 0.959729
iteration: 1100 sigma/SD(Y): 0.962476
iteration: 1200 sigma/SD(Y): 0.963027
iteration: 1300 sigma/SD(Y): 0.963803
iteration: 1400 sigma/SD(Y): 0.963116
iteration: 1500 sigma/SD(Y): 0.960237
iteration: 1600 sigma/SD(Y): 0.957822
iteration: 1700 sigma/SD(Y): 0.960867
iteration: 1800 sigma/SD(Y): 0.955235
iteration: 1900 sigma/SD(Y): 0.956125
time for loo

BCF trust fit saved (n = 84603).

The saved fits persist correctly: each `$control_fit` / `$moderate_fit` carries a `str` serialized forest alongside the live pointer, and on reload `get_forest_fit()` rebuilds the pointer from that string, so the post-processing step can recover μ(x) and τ(x) from the `.rds`.

# Session Info

In [ ]:
session_info()


─ Session info ───────────────────────────────────────────────────────────────
 setting  value
 version  R version 4.5.3 (2026-03-11 ucrt)
 os       Windows 11 x64 (build 26200)
 system   x86_64, mingw32
 ui       RTerm
 language (EN)
 collate  English_United States.utf8
 ctype    English_United States.utf8
 tz       Europe/Berlin
 date     2026-06-12
 pandoc   3.8.3 @ c:\\Program Files\\Positron\\resources\\app\\quarto\\bin\\tools/ (via rmarkdown)
 quarto   1.9.36 @ C:\\PROGRA~1\\Quarto\\bin\\quarto.exe

─ Packages ───────────────────────────────────────────────────────────────────
 package       * version    date (UTC) lib source
 cli             3.6.5      2025-04-23 [1] CRAN (R 4.5.1)
 codetools       0.2-20     2024-03-31 [2] CRAN (R 4.5.3)
 data.table      1.18.0     2025-12-24 [1] CRAN (R 4.5.2)
 digest          0.6.39     2025-11-19 [1] CRAN (R 4.5.3)
 dplyr         * 1.1.4      2023-11-17 [1] CRAN (R 4.5.1)
 evaluate        1.0.5      2025-08-27 [1] CRAN (R 4.5.2)
 farver     

# Execution Time

In [ ]:
end_time <- Sys.time()
exec_time <- end_time - start_time
cat(paste(
  "R code execution time:",
  round(as.numeric(exec_time, units = "secs"), 2),
  "seconds."
))


R code execution time: 12011.93 seconds.

```` markdown
---
subtitle: |
  Hierarchical BCF Fits on Real Data (Dictator + Trust)
date: last-modified
date-format: MMMM D, YYYY
format:
  html:
    toc: true
    toc-depth: 3
    code-fold: true
    code-tools: true
execute:
  echo: true
  warning: true
  eval: true
  message: true
---

# Purpose {#sec-purpose}

This notebook fits the **robustness** estimator — hierarchical BCF via the local `multibart` package — on the full real data for both game outcomes.
Scope is **fitting and saving only**: each fit is written to `data/03_final/` for the later post-processing step.
No τ(x) extraction, projection, variance-component summaries, figures, or tables are produced here.

Following the resolved feasibility gate, the model uses **country random intercepts only** (the nested respondent-within-country fit was benchmarked and found infeasible; see `03_multibart_nested_ri_test.qmd`, `#sec-purpose`).
Within-respondent dependence across the three conjoint rounds is left for a respondent cluster bootstrap in the post-processing step, not modelled here.
The variable definitions are reused verbatim from the validated real-data section of `03_multibart_nested_ri_test.qmd` (`#sec-subsample`); the only change is fitting on the full per-game sample rather than a 10,000-row subsample.

::: callout-note
The full-sample BCF fits are the expensive step (each game is ~85k rows with `nburn + nsim = 2000`).
Quarto `freeze: auto` caches execution at the file level.
Do **not** add knitr per-chunk `cache: true` here: the fitted forest's `tree_samples` is a live Rcpp external pointer that does not survive cache serialize/reload.
:::

# Setup {#sec-setup}

The setup chunk loads the project packages plus `multibart`'s own dependencies (`Rcpp`, `RcppArmadillo`, `Rcereal`, `fastDummies`) before the local package is touched.

quarto-executable-code-5450563D

```r
#| label: setup
#| include: false
# execution time
start_time <- Sys.time()
# console width
options(width = 80)
# packages
p_required <- c(
  "tidyverse", # dplyr, ggplot2, tidyr
  "here", # relative paths
  "sessioninfo", # session docs

  # multibart dependencies -----------------------------------------------------
  "Rcpp", # multibart Depends + LinkingTo
  "RcppArmadillo", # multibart Depends + LinkingTo
  "Rcereal", # multibart Depends + LinkingTo
  "fastDummies" # multibart helper dependency
)
packages <- rownames(installed.packages())
p_to_install <- p_required[!(p_required %in% packages)]
if (length(p_to_install) > 0) {
  install.packages(p_to_install)
}
sapply(p_required, require, character.only = TRUE)
rm(p_required, p_to_install, packages)
```

## Local multibart package {#sec-multibart}

`multibart` lives only as source under `code/multibart`; it is installed once (compiling its C++) then attached with `library()`, which is robust for both interactive use and `quarto render`.

quarto-executable-code-5450563D

```r
#| label: setup-multibart
#| include: false
# install local source on first use
if (!requireNamespace("multibart", quietly = TRUE)) {
  install.packages(
    here::here("code", "multibart"),
    repos = NULL,
    type = "source"
  )
}
# attach installed package
library(multibart)
```

# Load data {#sec-load}

quarto-executable-code-5450563D

```r
#| label: load-data
dat <- readRDS(here::here("data", "02_processed", "eu25_long.rds"))
```

# Variable definitions {#sec-vars}

The frozen moderator list: profile-level conjoint attributes plus respondent-level pre-treatment items (items and indices together), exactly as in `03_multibart_nested_ri_test.qmd`.

quarto-executable-code-5450563D

```r
#| label: moderators
prof_mods <- c(
  "cj_nationality_shown",
  "der_cj_eu_identity",
  "der_cj_partisanship",
  "cj_age",
  "cj_sex",
  "cj_class"
)
resp_mods <- c(
  "q_gender",
  "q_age",
  "q_identity_country",
  "q_identity_eu",
  "q_identity_europe",
  "q_religion",
  "q_class",
  "q_eu_efficacy_understand",
  "q_pop_reps",
  "q_pop_goodevil",
  "q_pop_compromise",
  "q_dem_compromise",
  "q_dem_listen",
  "q_tech_experts",
  "q_tech_leaders",
  "q_party_harm",
  "q_people_incompetent",
  "q_eu_longterm",
  "q_eu_responsive",
  "q_eu_imp_nat_econ",
  "q_eu_imp_nat_cul",
  "q_eu_imp_nat_pol",
  "q_eu_abolish",
  "q_eu_satisfaction",
  "q_rural_urban",
  "q_edu_age_stop"
)
```

`prep_game()` assembles the design for one game: outcome, treatment, IDs, and the numeric moderator matrix.

quarto-executable-code-5450563D

```r
#| label: prep-game
prep_game <- function(dat, game) {
  sub <- dat |>
    filter(cj_game_type == game) # full per-game sample (no subsampling)

  y <- sub$cj_pl2
  z <- as.integer(sub$cj_rel == "muslim") # Muslim vs all-else pooled
  country_id <- factor(sub$meta_country)
  respondent_id <- factor(sub$meta_pid)

  mod_df <- sub |>
    select(all_of(c(prof_mods, resp_mods))) |>
    mutate(across(where(is.character), factor))
  sub_X <- model.matrix(~ . - 1, data = mod_df) # factors -> dummies

  stopifnot(
    !anyNA(sub_X),
    all(z %in% c(0L, 1L)),
    !anyNA(y),
    nrow(sub_X) == length(y)
  )

  list(
    y = y,
    z = z,
    sub_X = sub_X,
    country_id = country_id,
    respondent_id = respondent_id
  )
}
```

`fit_bcf()` fits one country-only random-intercept BCF: the `n x C` country-dummy design is the random-effects block, with a single variance component and known conjoint propensity.

quarto-executable-code-5450563D

```r
#| label: fit-bcf
fit_bcf <- function(d) {
  # country-only random-intercept design, built inline
  # (the random_intercepts() helper has a known typo bug; get_dummies() is fine)
  country_dummies <- get_dummies(d$country_id) # n x C
  C <- ncol(country_dummies)
  Q_country <- matrix(1, nrow = C, ncol = 1) # single variance component

  stopifnot(
    nrow(country_dummies) == length(d$y),
    C == nlevels(d$country_id),
    C > 1 # else sampler disables REs
  )

  bcf_continuous_linear(
    y = d$y,
    z = d$z,
    x_control = d$sub_X, # numeric model matrix of moderators
    x_moderate = d$sub_X,
    zhat = rep(0.25, length(d$y)), # known conjoint propensity
    randeff_design = country_dummies,
    randeff_variance_component_design = Q_country,
    randeff_scales = 1, # length K = 1 (one component)
    randeff_df = 3,
    nburn = 1000,
    nsim = 1000,
    update_interval = 100
  )
}
```

# Dictator-game fit {#sec-dictator}

quarto-executable-code-5450563D

```r
#| label: fit-dictator
d_dict <- prep_game(dat, "cj_dict")
fit_dict <- fit_bcf(d_dict)
saveRDS(
  fit_dict,
  here::here("data", "03_final", "bcf_dictator.rds"),
  compress = "xz"
)
cat(sprintf("BCF dictator fit saved (n = %d).\n", length(d_dict$y)))
```

# Trust-game fit {#sec-trust}

quarto-executable-code-5450563D

```r
#| label: fit-trust
d_trust <- prep_game(dat, "cj_trust")
fit_trust <- fit_bcf(d_trust)
saveRDS(
  fit_trust,
  here::here("data", "03_final", "bcf_trust.rds"),
  compress = "xz"
)
cat(sprintf("BCF trust fit saved (n = %d).\n", length(d_trust$y)))
```

The saved fits persist correctly: each `$control_fit` / `$moderate_fit` carries a `str` serialized forest alongside the live pointer, and on reload `get_forest_fit()` rebuilds the pointer from that string, so the post-processing step can recover μ(x) and τ(x) from the `.rds`.

# Session Info {#sec-session-info .appendix .unnumbered}

quarto-executable-code-5450563D

```r
#| label: session-info
#| eval: true
#| echo: true
session_info()
```

# Execution Time {#sec-exec-time .appendix .unnumbered}

quarto-executable-code-5450563D

```r
#| label: exec-time
#| eval: true
#| echo: true
#| include: true
end_time <- Sys.time()
exec_time <- end_time - start_time
cat(paste(
  "R code execution time:",
  round(as.numeric(exec_time, units = "secs"), 2),
  "seconds."
))
```
````